In [0]:
import pandas as pd

In [0]:
base_path = "/Volumes/workspace/olist_raw/bronze/"

In [0]:
orders = pd.read_csv(base_path + "olist_orders_dataset.csv")

In [0]:
orders = pd.read_csv(base_path + "olist_orders_dataset.csv")
order_items = pd.read_csv(base_path + "olist_order_items_dataset.csv")
order_payments = pd.read_csv(base_path + "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(base_path + "olist_order_reviews_dataset.csv")
customers = pd.read_csv(base_path + "olist_customers_dataset.csv")
sellers = pd.read_csv(base_path + "olist_sellers_dataset.csv")
products = pd.read_csv(base_path + "olist_products_dataset.csv")
category_translation = pd.read_csv(base_path + "product_category_name_translation.csv")
geolocation = pd.read_csv(base_path + "olist_geolocation_dataset.csv")

print("Tabelas carregadas com sucesso")

In [0]:
tabelas = {
    "orders": orders,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "customers": customers,
    "sellers": sellers,
    "products": products,
    "category_translation": category_translation,
    "geolocation": geolocation
}

for nome, df in tabelas.items():
    print(f"\n{'='*50}")
    print(f"Tabela: {nome}")
    print(f"Linhas: {df.shape[0]} | Colunas: {df.shape[1]}")
    print(f"\nTipos de dados:")
    print(df.dtypes)
    print(f"\nNulos por coluna:")
    print(df.isnull().sum())

In [0]:
print("orders → customers:", 
      orders["customer_id"].isin(customers["customer_id"]).all())

print("order_items → orders:", 
      order_items["order_id"].isin(orders["order_id"]).all())

print("order_payments → orders:", 
      order_payments["order_id"].isin(orders["order_id"]).all())

print("order_reviews → orders:", 
      order_reviews["order_id"].isin(orders["order_id"]).all())

print("order_items → products:", 
      order_items["product_id"].isin(products["product_id"]).all())

print("order_items → sellers:", 
      order_items["seller_id"].isin(sellers["seller_id"]).all())

In [0]:
print("=== INCONSISTÊNCIAS IDENTIFICADAS ===")
print("\n[orders]")
print(f"  - Datas armazenadas como string (object) — precisam virar datetime na Silver")
print(f"  - order_approved_at: {orders['order_approved_at'].isnull().sum()} nulos (pedidos não aprovados)")
print(f"  - order_delivered_carrier_date: {orders['order_delivered_carrier_date'].isnull().sum()} nulos (não coletados)")
print(f"  - order_delivered_customer_date: {orders['order_delivered_customer_date'].isnull().sum()} nulos (não entregues)")

print("\n[order_reviews]")
print(f"  - review_comment_title: {order_reviews['review_comment_title'].isnull().sum()} nulos (clientes sem título)")
print(f"  - review_comment_message: {order_reviews['review_comment_message'].isnull().sum()} nulos (clientes sem comentário)")

print("\n[products]")
print(f"  - product_category_name: {products['product_category_name'].isnull().sum()} nulos (produtos sem categoria)")

print("\n[geolocation]")
print(f"  - {geolocation.shape[0]:,} linhas — múltiplas coordenadas por CEP, precisará agregação para Q3")

## Phase 1 — Findings Summary

### Dataset Overview
| Table | Rows | Columns |
|---|---|---|
| orders | 99,441 | 8 |
| order_items | 112,650 | 7 |
| order_payments | 103,886 | 5 |
| order_reviews | 99,224 | 7 |
| customers | 99,441 | 5 |
| sellers | 3,095 | 4 |
| products | 32,951 | 9 |
| category_translation | 71 | 2 |
| geolocation | 1,000,163 | 5 |

### Referential Integrity
All foreign keys validated — True across all 6 relationships checked:
- orders → customers ✅
- order_items → orders ✅
- order_payments → orders ✅
- order_reviews → orders ✅
- order_items → products ✅
- order_items → sellers ✅

### Inconsistencies Identified

**[orders]**
- 4 date columns stored as string (object) — need datetime conversion in Silver layer
- `order_approved_at`: 160 nulls — orders not approved
- `order_delivered_carrier_date`: 1,783 nulls — orders not collected by carrier
- `order_delivered_customer_date`: 2,965 nulls — orders not delivered to customer

**[order_reviews]**
- `review_comment_title`: 87,656 nulls — customers did not fill title
- `review_comment_message`: 58,247 nulls — customers did not leave a comment
- Note: `review_score` has 0 nulls — safe to use for Q1

**[products]**
- `product_category_name`: 610 nulls — products without category assignment

**[geolocation]**
- 1,000,163 rows — multiple coordinates per zip code
- Aggregation required before joining with orders for Q3